<div style="background: linear-gradient(135deg, #1e293b 0%, #0f172a 100%); padding: 30px; border-radius: 8px; margin-bottom: 25px;">
    <h1 style="color: #f8fafc; font-family: 'Segoe UI', sans-serif; font-weight: 700; margin: 0; font-size: 2.2em;">📚 Pipeline multi-edición PISA (2015+)</h1>
    <p style="color: #94a3b8; font-family: 'Segoe UI', sans-serif; margin: 8px 0 0 0; font-size: 1.1em;">Procesamiento de los tres dominios (Matemáticas, Lectura, Ciencias) + EDA</p>
</div>

# Descripción

Pipeline que construye las **tablas maestras** de PISA 2012, 2015, 2018 y 2022 (2025 queda fuera por ahora).

* **Países**: los 33 miembros OCDE presentes en todas las ediciones, con la **muestra completa** y los pesos originales de la OCDE.
* **Tablas**: por edición, una de **estudiantes** y otra de **colegios**, con **todas las columnas** originales, **sin imputar**, en `data/intermediate/datos_anuales/`.
* **PV**: de MATH, READ y SCIE (`PV1..PV10`; 5 en 2012).
* **Índices reescalados**: se añaden `ESCS_TREND`, `HISEI_TREND`, `HOMEPOS_TREND` y `PAREDINT_TREND` desde `escs_trend.csv`.
* **Flujo**: 1) procesar las ediciones → 2) validar las tablas → 3) generar el diccionario de columnas. El EDA está en `01_EDA_por_edicion.ipynb`.
* La lógica vive en `src/pisa_pipeline.py`.

## Diccionario de columnas

Cada edición tiene **dos tablas** (`pisa<año>_estudiantes.parquet`, una fila por alumno, y `pisa<año>_colegios.parquet`, una por escuela) con **todas las columnas originales de la OCDE** (≈630-1.280 en estudiantes, ≈200-430 en colegios); aquí solo se documentan las que usa el pipeline. El diccionario completo, con etiquetas, está en `data/intermediate/diccionario_columnas.csv`. Las columnas que una edición llama distinto (2012, 2015, 2022) se renombran. **Claves**: alumno `(CNT, CNTSTUID)` (en 2012 el id solo es único dentro del país), centro `(CNT, CNTSCHID)`.

**Identificadores y pesos**

| Columna | Descripción |
|---|---|
| `CNT` | Código de país. |
| `CNTSCHID` / `CNTSTUID` | Id de centro / de alumno (en 2012 `SCHOOLID` / `StIDStd`). |
| `STRATUM` | Estrato del diseño muestral de PISA. |
| `W_FSTUWT` | Peso final del alumno (a cuántos alumnos de la población representa). |
| `SENWT` | Peso "senado": cada país suma 5.000. En 2012 no existe y lo calcula el pipeline. |
| `PV1..PV10` × `MATH`/`READ`/`SCIE` | Valores plausibles (5 en 2012): estimaciones múltiples de la habilidad; se promedian, nunca se usa uno solo. |

**Contexto del estudiante** (`COLUMNAS_CONTEXTO_ESTUDIANTE`)

| Columna | Descripción |
|---|---|
| `ESCS`, `HISEI`, `PAREDINT`, `HOMEPOS` | Estatus económico, social y cultural; ocupación más alta de los padres (0-100); estudios más altos de los padres (años); posesiones del hogar. 2012 y 2015: `hisei` y `PARED`. |
| `ESCS_TREND`, `HISEI_TREND`, `HOMEPOS_TREND`, `PAREDINT_TREND` | Versiones reescaladas por la OCDE (`escs_trend.csv`) de 2012-2018, comparables con 2022. En 2022 son copia de las originales. |
| `WEALTH`, `HEDRES`, `CULTPOSS` | Riqueza familiar; recursos educativos; posesiones culturales. No existen en 2022 (`CULTPOSS` en 2012: `CULTPOS`). |
| `ICTRES` | Recursos TIC en casa. No existe en 2012; en 2022 faltan 19 de 33 países. |
| `ANXMAT`, `MATHEFF` | Ansiedad y autoeficacia en matemáticas (2012 y 2022). |
| `BELONG`, `TEACHSUP` | Pertenencia al centro; apoyo del profesorado. `TEACHSUP` mide distinta asignatura según la edición. |
| `BEINGBULLIED` | Acoso sufrido (2018; 2022 como `BULLIED`, renombrada). |
| `DISCLIMA` | Clima disciplinario. Distinta asignatura según la edición; solo se rellena en 2012 y 2018. |
| `MATHINT`, `MATHBEH` | Solo existen en 2012 con otro nombre (`INTMAT`, `MATBEH`): no se rellenan. |
| `OUTHOURS` | Horas de estudio fuera del horario escolar (2012 y 2015). |
| `ICTWKDY`, `ICTWKEND`, `ICTDISTR` | Uso TIC entre semana y fin de semana; malestar online (solo 2022; 7 de 33 países sin dato). |
| `SKIPPING`, `TARDYSD` | Absentismo e impuntualidad (solo 2022). |

**Contexto del centro** (`COLUMNAS_CONTEXTO_COLEGIO`)

| Columna | Descripción |
|---|---|
| `STRATIO`, `CLSIZE` | Alumnos por profesor; tamaño de clase. |
| `PROPMATH`, `SCHLCLI`, `EDUSHORT`, `STAFFSHORT` | Profesores de matemáticas certificados; clima escolar; escasez de recursos y de personal (`STAFFSHORT` no existe en 2012). |
| `SCHLTYPE` | Titularidad: 1 privado independiente, 2 privado concertado, 3 público. Sin dato en BEL, IRL, NOR, SWE (2018) y AUT, IRL, ISR, NOR (2022). |
| `W_SCHGRNRABWT` | Peso del centro (2012: `W_FSCHWT`). |

**Añadidas por el pipeline**: `EDICION` y `DOMINIO_PRINCIPAL` (año y dominio focal del ciclo).


⚠️ **Comparabilidad entre ciclos:** para comparar índices de contexto (ESCS, HOMEPOS...) *entre* ediciones y no solo dentro de cada una, necesitamos tener en cuenta que la OCDE publica índices reescalados específicos para análisis de tendencias (*trend indices*), porque la escala IRT/WLE de los índices derivados puede no ser directamente comparable de un ciclo a otro.

## 0. Configuración

In [1]:
import sys, os, gc
sys.path.append(os.path.abspath('../../'))   # para importar `src`

# recarga src/ al editarlo, sin reiniciar el kernel
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import pyarrow.parquet as pq

from src.pisa_pipeline import (
    EDICIONES_PISA, DOMINIOS, PAISES_OCDE_TODAS_LAS_EDICIONES,
    procesar_edicion, rutas_datos_anuales, columnas_pv,
    validar_tablas, generar_diccionario_columnas,
)

from src.utils_pisa import establecer_semilla

establecer_semilla()

✅ Semillas del proyecto establecidas con éxito (seed=42).


Cada edición se espera en `data/raw/<año>/` (ficheros de estudiantes y de centros en `.sas7bdat` o `.sav`; patrones en `EDICIONES_PISA`). 2012 viene como `.txt` + `.sas`: conviértelo antes con `src/pisa_txt_a_sav.py`. Para añadir una edición, inclúyela en `EDICIONES_A_PROCESAR` y en `EDICIONES_PISA`.

In [3]:
EDICIONES_A_PROCESAR = [2012, 2015, 2018, 2022]   # 2025 queda fuera por ahora
PAISES = PAISES_OCDE_TODAS_LAS_EDICIONES        # 33 países OCDE; None = todos
RUTA_RAW_BASE = '../../data/raw'                # data/raw/<año>/<fichero>
RUTA_INTERMEDIOS = '../../data/intermediate'    # -> datos_anuales/
RUTA_ESCS_TREND = '../../data/raw/escs_trend.csv'   # índices reescalados de la OCDE (2012-2018)

# None = muestra completa con pesos originales. Un valor en (0, 1) submuestrearía
# escuelas por estrato (con la semilla) y añadiría pesos ajustados (*_adj).
FRACCION_SUBMUESTRA = None
SEMILLA = 42

# Si los parquets de datos_anuales/ ya existen se reutilizan; pon True si cambias
# PAISES o FRACCION_SUBMUESTRA, o para forzar el reprocesamiento.
FORZAR_REPROCESAMIENTO = False

## 1. Procesamiento por edición

Por edición: filtro de países (`PAISES`) y lectura de **dos tablas separadas** (estudiantes y colegios) con **todas las columnas** y los **pesos originales** (muestra completa). Se guardan sin imputar en `data/intermediate/datos_anuales/`; si ya existen, se cargan sin releer el origen (`FORZAR_REPROCESAMIENTO`).

* **Países**: los 33 miembros OCDE en 2012 con datos en 2022. Se descartan LVA, LTU, COL y CRI (entraron después) y LUX (miembro, pero sin datos en 2022).
* **Índices reescalados**: se añaden `ESCS_TREND`, `HISEI_TREND`, `HOMEPOS_TREND` y `PAREDINT_TREND` desde `escs_trend.csv` (comparables con 2022). Los originales conservan su nombre. En 2022, que no está en el csv, las `_TREND` son copia de las originales.
* **Avisos**: el pipeline avisa de las columnas de contexto que una edición no trae (p. ej. `ICTRES` en 2012). No es un error.
* Los pesos replicados (`W_FSTURWT1..80`; `W_FSTR1..80` en 2012) son los originales y valen para errores estándar.

In [4]:
dimensiones = {}   # forma de las tablas guardadas

for año in EDICIONES_A_PROCESAR:
    print(f'--- Procesando PISA {año} (dominio principal: {EDICIONES_PISA[año]["dominio_principal"]}) ---')

    # Lee el origen (o los parquets existentes), filtra países, añade *_TREND y guarda en datos_anuales/
    df_estudiantes, df_colegios = procesar_edicion(
        año=año,
        ruta_raw=os.path.join(RUTA_RAW_BASE, str(año)),
        ruta_intermedios=RUTA_INTERMEDIOS,
        fraccion=FRACCION_SUBMUESTRA,
        seed=SEMILLA,
        paises=PAISES,
        usar_checkpoint_si_existe=not FORZAR_REPROCESAMIENTO,
        ruta_escs_trend=RUTA_ESCS_TREND,
    )
    dimensiones[año] = {
        'alumnos': len(df_estudiantes), 'cols alumnos': df_estudiantes.shape[1],
        'centros': len(df_colegios), 'cols centros': df_colegios.shape[1],
        'países': df_estudiantes['CNT'].nunique(),
    }
    print(f'Estudiantes: {df_estudiantes.shape} | Colegios: {df_colegios.shape} | '
          f'{df_estudiantes["CNT"].nunique()} países\n')
    del df_estudiantes, df_colegios   # la tabla completa ya está en parquet
    gc.collect()

--- Procesando PISA 2012 (dominio principal: MATH) ---
  Parquets anuales ya existen, se cargan sin reprocesar el origen: ../../data/intermediate\datos_anuales


Estudiantes: (290158, 641) | Colegios: (11725, 292) | 33 países



--- Procesando PISA 2015 (dominio principal: SCIE) ---
  Parquets anuales ya existen, se cargan sin reprocesar el origen: ../../data/intermediate\datos_anuales


Estudiantes: (238452, 927) | Colegios: (9076, 274) | 33 países



--- Procesando PISA 2018 (dominio principal: READ) ---
  Parquets anuales ya existen, se cargan sin reprocesar el origen: ../../data/intermediate\datos_anuales


Estudiantes: (269587, 1125) | Colegios: (10366, 199) | 33 países



--- Procesando PISA 2022 (dominio principal: MATH) ---


  Aviso PISA 2022: columnas no encontradas en el fichero de estudiantes: ['WEALTH', 'CULTPOSS', 'HEDRES', 'MATHINT', 'MATHBEH', 'DISCLIMA', 'OUTHOURS']


  Aviso PISA 2022: columnas no encontradas en el fichero de centros: ['SCHLCLI']


Estudiantes: (268610, 1284) | Colegios: (10131, 432) | 33 países



In [5]:
pd.DataFrame(dimensiones)   # resumen de las tablas maestras

,2012,2015,2018,2022
alumnos,290158,238452,269587,268610
cols alumnos,641,927,1125,1284
centros,11725,9076,10366,10131
cols centros,292,274,199,432
países,33,33,33,33


## 2. Validación de las tablas maestras

Con `assert`, para cada edición: están los 33 países; no hay alumnos ni centros duplicados ni ids nulos; cada alumno tiene su centro y cada centro algún alumno; los pesos son positivos; no faltan PV; y `SENWT` suma 5.000 por país.

**Claves**: alumno = `(CNT, CNTSTUID)`; centro = `(CNT, CNTSCHID)`. En 2012 `CNTSTUID` solo es único **dentro del país** (en 2015-2022 incluye el prefijo del país), por eso nunca se usa solo.

In [6]:
columnas_clave_est = ['CNT', 'CNTSCHID', 'CNTSTUID', 'W_FSTUWT', 'SENWT'] + [c for d in DOMINIOS for c in columnas_pv(d)]
resultados = {}

for año in EDICIONES_A_PROCESAR:
    ruta_est, ruta_col = rutas_datos_anuales(RUTA_INTERMEDIOS, año)
    presentes = [c for c in columnas_clave_est if c in pq.ParquetFile(ruta_est).schema.names]
    est = pd.read_parquet(ruta_est, columns=presentes)
    col = pd.read_parquet(ruta_col, columns=['CNT', 'CNTSCHID', 'W_SCHGRNRABWT'])
    resultados[año] = validar_tablas(est, col, PAISES)['valor']   # falla si algo no cuadra

pd.DataFrame(resultados)

,2012,2015,2018,2022
países distintos a los pedidos,0.000000e+00,0.00000,0.00000,0.00000
"alumnos duplicados (CNT, CNTSTUID)",0.000000e+00,0.00000,0.00000,0.00000
"centros duplicados (CNT, CNTSCHID)",0.000000e+00,0.00000,0.00000,0.00000
ids nulos,0.000000e+00,0.00000,0.00000,0.00000
alumnos sin centro en la tabla de colegios,0.000000e+00,0.00000,0.00000,0.00000
centros sin alumnos,0.000000e+00,0.00000,0.00000,0.00000
pesos de alumno nulos o <= 0,0.000000e+00,0.00000,0.00000,0.00000
pesos de centro nulos o <= 0,0.000000e+00,0.00000,0.00000,0.00000
valores plausibles nulos,0.000000e+00,0.00000,0.00000,0.00000
desvío máximo de SENWT respecto a 5000 por país,9.094947e-13,0.00376,0.00304,0.00605


## 3. Diccionario de columnas (CSV)

Las tablas tienen hasta 1.280 columnas y los parquets no guardan sus etiquetas. Se genera `data/intermediate/diccionario_columnas.csv` (edición, tabla, columna, nombre original, etiqueta) con los metadatos de los ficheros originales y los nombres ya renombrados, incluidas las columnas que añade el pipeline. Es pequeño y se versiona en git.

In [7]:
ruta_diccionario = os.path.join(RUTA_INTERMEDIOS, 'diccionario_columnas.csv')
diccionario_columnas = generar_diccionario_columnas(RUTA_RAW_BASE, EDICIONES_A_PROCESAR, ruta_diccionario)
print(f'{len(diccionario_columnas)} filas -> {ruta_diccionario}')

# Ejemplo de uso: buscar columnas por palabra de la etiqueta
diccionario_columnas[diccionario_columnas['ETIQUETA'].str.contains('bullied', case=False)].head(10)

5174 filas -> ../../data/intermediate\diccionario_columnas.csv


,EDICION,TABLA,COLUMNA,NOMBRE_ORIGINAL,ETIQUETA
720,2012,colegios,SC22Q08,SC22Q08,Learning Hindrance - Students being bullied
2445,2018,estudiantes,ST207Q01HA,ST207Q01HA,Agree: It irritates me when nobody defends bul...
2448,2018,estudiantes,ST207Q04HA,ST207Q04HA,Agree: I feel bad seeing other students bullied.
2449,2018,estudiantes,ST207Q05HA,ST207Q05HA,Agree: I like it when someone stands up for ot...
3049,2018,estudiantes,BEINGBULLIED,BEINGBULLIED,Student's experience of being bullied (WLE)
4449,2022,estudiantes,BEINGBULLIED,BULLIED,Being bullied (WLE)


## ✅ Próximos pasos

* **Tablas maestras** (2 por año, sin imputar, `datos_anuales/`): hechas y validadas. El EDA está en `01_EDA_por_edicion.ipynb`.
* **Foco actual**: deberes (tu compañero) y presupuesto por alumno (nuevo notebook de EDA del presupuesto con datos externos de gasto), antes de modelar.
* **Comparabilidad entre ediciones** (informe técnico de PISA 2022, cap. 19): los índices *trend* de 2022 se enlazaron con un ciclo anterior (2012: `ANXMAT`, `MATHEFF`, `TEACHSUP`, `DISCLIM`; 2018: `BULLIED`→`BEINGBULLIED`, `BELONG`, `EDUSHORT`, `STAFFSHORT`) y solo esos pares están respaldados. `ESCS` y sus componentes tienen versión reescalada (`*_TREND`); `HOMEPOS_TREND` (+0,30 a +0,32) no enlaza bien con el `HOMEPOS` de 2022 (+0,06). El resto de índices, dentro de cada edición. Para comparar medias entre ediciones, ponderar con `SENWT`.
* **Alias pendientes** (decisión): `DISCLIM`→`DISCLIMA` y `TEACHSUP` en 2022 daría pares 2012-2022 en matemáticas. `MATHINT` y `MATHBEH` solo existen en 2012 (`INTMAT`, `MATBEH`).
* **`SCHLTYPE`**: sin dato en 4 países en 2018 y 4 en 2022; revisar en el codebook si otra pregunta de centro permite reconstruirlo.